# Exhaust Manifolds

These are 3D printable exhaust manifolds which connects a muffler tube to exhaust tips. There are 2 manifolds, one driver and one passenger. The manifolds are further subdivided into right and left sides so that inner supports can be accessed after 3D printing. Each side includes a guide so that the subassemblies can be aligned.

The outer diameter of the tubing is 2.5", and the inlets are connected to midpipes with slide on clamps. The exhaust tips have their own clamps which fit over the tubing. 

## Creating the manifolds

The data looks correct, so now we need to build the manifold in cadquery. To support splitting the design into multiple 3D printing meshes, the function allows variable sweep and trim profiles. 

In [22]:
%load_ext autoreload
%autoreload 2

from build import Builder
from build import Logger
import cadquery as cq
from jupyter_cadquery import *
import logging

logging.basicConfig(filename="out.txt", level=logging.DEBUG, filemode="w")

logger = Logger()
builder = Builder(logger=logger)

driver_wire, passenger_wire = builder.create_wire("driver"), builder.create_wire("passenger")
show(driver_wire, passenger_wire)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


HTML(value='⏳ <b>Building...</b>')

CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

Build the manifolds so we can perform some basic verification of the shapes.

In [23]:
logger = Logger()
builder = Builder(logger=logger)

driver_manifold, passenger_manifold = builder.build_tube("driver"), builder.build_tube("passenger")

assy = cq.Assembly()
assy.add(driver_manifold, name="driver_manifold", color="red")
assy.add(passenger_manifold, name="passenger_manifold", color="green")
show(assy)

HTML(value='⏳ <b>Building...</b>')

cc


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

Build individual parts.

In [26]:
logger = Logger()
builder = Builder(logger=logger)

assy = cq.Assembly()
assy.add(builder.build_part("driver"), name="driver_left", color="red")
assy.add(builder.build_part("driver", right=True), name="driver_right", color="green")
assy.add(builder.build_part("passenger"), name="passenger_left", color="purple")
assy.add(builder.build_part("passenger", right=True), name="passenger_right", color="yellow")
show(assy)

_, path_obj = builder.create_wire("driver")
pos = path_obj.positionAt(0)
tan = path_obj.tangentAt(0)
probe_plane = cq.Plane(origin=pos, xDir=cq.Vector(0, 0, 1), normal=tan)
cutting_plate = cq.Workplane(probe_plane).rect(500, 500).extrude(0.01)
show(builder.build_part("driver").intersect(cutting_plate))

HTML(value='⏳ <b>Building...</b>')

cccc


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

As a sanity check, show that we can recombine the parts back into the original manifold shape.

In [25]:
logger = Logger()
builder = Builder(logger=logger)

driver_manifold, driver_manifold_from_parts = builder.build_back_manifold("driver")
passenger_manifold, passenger_manifold_from_parts = builder.build_back_manifold("passenger")

assy = cq.Assembly()
assy.add(driver_manifold, name="driver_manifold", color="red")
assy.add(driver_manifold_from_parts, name="driver_manifold_from_parts", color="blue")
assy.add(passenger_manifold, name="passenger_manifold", color="green")
assy.add(passenger_manifold_from_parts, name="passenger_manifold_from_parts", color="yellow")
show(assy)

HTML(value='⏳ <b>Building...</b>')

cccc


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=600, id…

Display the error for each part.

In [ ]:
logger = Logger()
builder = Builder(logger=logger)

for name in builder.names:
    error_pct = builder.calc_part_error(name)
    logger.print(f"{name} error: {round(error_pct)}%")